# LAB 3 — Auto Loader Monitoring

This notebook runs isolated Auto Loader experiments to analyze files per batch and compare `availableNow` with `once`.

## 1. Load shared configuration

In [0]:
%run ./lab03_config

## 2. Validate the benchmark source files

In [0]:
benchmark_source_path = staging_initial_path

benchmark_files = [
    file_info
    for file_info in dbutils.fs.ls(benchmark_source_path)
    if file_info.name.endswith(".json")
]

if not benchmark_files:
    raise FileNotFoundError(
        f"No benchmark JSON files found: {benchmark_source_path}"
    )

print(f"Benchmark source files: {len(benchmark_files)}")

## 3. Define the benchmark runner

In [0]:
from pyspark.sql.functions import col, current_timestamp

def run_autoloader_benchmark(
    test_name: str,
    files_per_trigger: int,
    test_trigger: str
) -> list[dict]:
    safe_name = test_name.replace("-", "_")

    test_schema_path = (
        f"{volume_root}/system/schema/monitoring/{safe_name}"
    )
    test_checkpoint_path = (
        f"{volume_root}/system/checkpoints/monitoring/{safe_name}"
    )
    test_table = (
        f"{catalog}.{schema}.lab03_monitoring_{safe_name}"
    )

    dbutils.fs.rm(test_schema_path, recurse=True)
    dbutils.fs.rm(test_checkpoint_path, recurse=True)
    spark.sql(f"DROP TABLE IF EXISTS {test_table}")

    test_df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option(
            "cloudFiles.schemaLocation",
            test_schema_path
        )
        .option(
            "cloudFiles.maxFilesPerTrigger",
            files_per_trigger
        )
        .option("cloudFiles.inferColumnTypes", "true")
        .load(benchmark_source_path)
        .withColumn(
            "_source_file",
            col("_metadata.file_path")
        )
        .withColumn(
            "_ingested_at",
            current_timestamp()
        )
    )

    writer = (
        test_df.writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            test_checkpoint_path
        )
        .queryName(f"lab03_monitoring_{safe_name}")
    )

    if test_trigger == "availableNow":
        query = (
            writer
            .trigger(availableNow=True)
            .toTable(test_table)
        )
    elif test_trigger == "once":
        query = (
            writer
            .trigger(once=True)
            .toTable(test_table)
        )
    else:
        raise ValueError(
            f"Unsupported benchmark trigger: {test_trigger}"
        )

    query.awaitTermination()

    rows = []

    for progress in query.recentProgress:
        source = (
            progress.get("sources", [{}])[0]
            if progress.get("sources")
            else {}
        )
        metrics = source.get("metrics", {})

        rows.append(
            {
                "test_name": test_name,
                "trigger_type": test_trigger,
                "max_files_per_trigger": files_per_trigger,
                "batch_id": progress.get("batchId"),
                "num_input_rows": progress.get("numInputRows"),
                "input_rows_per_second": progress.get(
                    "inputRowsPerSecond"
                ),
                "processed_rows_per_second": progress.get(
                    "processedRowsPerSecond"
                ),
                "trigger_execution_ms": (
                    progress.get("durationMs", {})
                    .get("triggerExecution")
                ),
                "num_files_outstanding": metrics.get(
                    "numFilesOutstanding"
                ),
                "num_bytes_outstanding": metrics.get(
                    "numBytesOutstanding"
                ),
            }
        )

    processed_files = (
        spark.table(test_table)
        .select("_source_file")
        .distinct()
        .count()
    )

    row_count = spark.table(test_table).count()

    summary = {
        "test_name": test_name,
        "trigger_type": test_trigger,
        "max_files_per_trigger": files_per_trigger,
        "batch_count": len(rows),
        "processed_files": processed_files,
        "output_rows": row_count,
    }

    return rows, summary

## 4. Compare files per trigger with `availableNow`

In [0]:
benchmark_progress = []
benchmark_summaries = []

for files_per_trigger in [10, 50, 200]:
    test_name = f"available_now_{files_per_trigger}"

    progress_rows, summary = run_autoloader_benchmark(
        test_name=test_name,
        files_per_trigger=files_per_trigger,
        test_trigger="availableNow"
    )

    benchmark_progress.extend(progress_rows)
    benchmark_summaries.append(summary)

print("availableNow benchmarks completed.")

## 5. Display files-per-trigger results

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType,
)


def to_int(value):
    if value is None:
        return None

    try:
        return int(value)
    except (TypeError, ValueError):
        return None


def to_float(value):
    if value is None:
        return None

    try:
        return float(value)
    except (TypeError, ValueError):
        return None


clean_progress = []

for row in benchmark_progress:
    clean_progress.append(
        {
            "test_name": str(row.get("test_name")),
            "trigger_type": str(row.get("trigger_type")),
            "max_files_per_trigger": to_int(
                row.get("max_files_per_trigger")
            ),
            "batch_id": to_int(
                row.get("batch_id")
            ),
            "num_input_rows": to_int(
                row.get("num_input_rows")
            ),
            "input_rows_per_second": to_float(
                row.get("input_rows_per_second")
            ),
            "processed_rows_per_second": to_float(
                row.get("processed_rows_per_second")
            ),
            "trigger_execution_ms": to_int(
                row.get("trigger_execution_ms")
            ),
            "num_files_outstanding": to_int(
                row.get("num_files_outstanding")
            ),
            "num_bytes_outstanding": to_int(
                row.get("num_bytes_outstanding")
            ),
        }
    )


clean_summaries = []

for row in benchmark_summaries:
    clean_summaries.append(
        {
            "test_name": str(row.get("test_name")),
            "trigger_type": str(row.get("trigger_type")),
            "max_files_per_trigger": to_int(
                row.get("max_files_per_trigger")
            ),
            "batch_count": to_int(
                row.get("batch_count")
            ),
            "processed_files": to_int(
                row.get("processed_files")
            ),
            "output_rows": to_int(
                row.get("output_rows")
            ),
        }
    )


progress_schema = StructType([
    StructField("test_name", StringType(), False),
    StructField("trigger_type", StringType(), False),
    StructField("max_files_per_trigger", LongType(), True),
    StructField("batch_id", LongType(), True),
    StructField("num_input_rows", LongType(), True),
    StructField("input_rows_per_second", DoubleType(), True),
    StructField("processed_rows_per_second", DoubleType(), True),
    StructField("trigger_execution_ms", LongType(), True),
    StructField("num_files_outstanding", LongType(), True),
    StructField("num_bytes_outstanding", LongType(), True),
])


summary_schema = StructType([
    StructField("test_name", StringType(), False),
    StructField("trigger_type", StringType(), False),
    StructField("max_files_per_trigger", LongType(), True),
    StructField("batch_count", LongType(), True),
    StructField("processed_files", LongType(), True),
    StructField("output_rows", LongType(), True),
])


if clean_progress:
    progress_df = spark.createDataFrame(
        clean_progress,
        schema=progress_schema
    )

    display(
        progress_df.orderBy(
            "max_files_per_trigger",
            "batch_id"
        )
    )


summary_df = spark.createDataFrame(
    clean_summaries,
    schema=summary_schema
)

display(
    summary_df.orderBy(
        "max_files_per_trigger"
    )
)

## 6. Compare `availableNow` and `once`

In [0]:
trigger_progress = []
trigger_summaries = []

for test_trigger in ["availableNow", "once"]:
    test_name = f"trigger_{test_trigger.lower()}"

    progress_rows, summary = run_autoloader_benchmark(
        test_name=test_name,
        files_per_trigger=50,
        test_trigger=test_trigger
    )

    trigger_progress.extend(progress_rows)
    trigger_summaries.append(summary)

print("Trigger comparison completed.")

## 7. Display trigger comparison

In [0]:
clean_trigger_progress = []

for row in trigger_progress:
    clean_trigger_progress.append(
        {
            "test_name": str(row.get("test_name")),
            "trigger_type": str(row.get("trigger_type")),
            "max_files_per_trigger": to_int(
                row.get("max_files_per_trigger")
            ),
            "batch_id": to_int(
                row.get("batch_id")
            ),
            "num_input_rows": to_int(
                row.get("num_input_rows")
            ),
            "input_rows_per_second": to_float(
                row.get("input_rows_per_second")
            ),
            "processed_rows_per_second": to_float(
                row.get("processed_rows_per_second")
            ),
            "trigger_execution_ms": to_int(
                row.get("trigger_execution_ms")
            ),
            "num_files_outstanding": to_int(
                row.get("num_files_outstanding")
            ),
            "num_bytes_outstanding": to_int(
                row.get("num_bytes_outstanding")
            ),
        }
    )


clean_trigger_summaries = []

for row in trigger_summaries:
    clean_trigger_summaries.append(
        {
            "test_name": str(row.get("test_name")),
            "trigger_type": str(row.get("trigger_type")),
            "max_files_per_trigger": to_int(
                row.get("max_files_per_trigger")
            ),
            "batch_count": to_int(
                row.get("batch_count")
            ),
            "processed_files": to_int(
                row.get("processed_files")
            ),
            "output_rows": to_int(
                row.get("output_rows")
            ),
        }
    )


trigger_summary_df = spark.createDataFrame(
    clean_trigger_summaries,
    schema=summary_schema
)

display(
    trigger_summary_df.orderBy("trigger_type")
)


if clean_trigger_progress:
    trigger_progress_df = spark.createDataFrame(
        clean_trigger_progress,
        schema=progress_schema
    )

    display(
        trigger_progress_df.orderBy(
            "trigger_type",
            "batch_id"
        )
    )

## 8. Interpret the results

In [0]:
available_now_result = next(
    item
    for item in trigger_summaries
    if item["trigger_type"] == "availableNow"
)

once_result = next(
    item
    for item in trigger_summaries
    if item["trigger_type"] == "once"
)

print(
    "availableNow processed "
    f"{available_now_result['processed_files']} files "
    f"across {available_now_result['batch_count']} batches."
)

print(
    "once processed "
    f"{once_result['processed_files']} files "
    f"across {once_result['batch_count']} batch."
)

print(
    "Conclusion: availableNow is suitable for clearing all currently "
    "available files across multiple micro-batches, while once executes "
    "a single triggered batch."
)

## 9. Final result

In [0]:
print("Auto Loader monitoring experiments completed.")
print("Next notebook: lab03_05_checkpoint_recovery")